# Tomahawk G2 — self-contained Run 2 diagnostic

This notebook already contains your exact failed Run 2 checkpoint (SHA-256 `1535aee65adc1db7411ae1fc2a33622ae34ca522ce3ca6dce40b461c02ccda85`). **Import this notebook in Kaggle and choose Run All. Do not attach any input dataset.** Enable Internet only if DESC 0.17.3 needs installation.

It does not rerun the four-hour solve. It compares the saved L12 and L16 boundary shapes to the CAD reference and writes `/kaggle/working/Tomahawk_G2_Run_2_Diagnostic.zip`. Download that ZIP and upload it here. Even if a Python step fails, the second cell writes the traceback into the ZIP. A kernel-level kill could still prevent output.


In [ ]:
from pathlib import Path
import base64
import hashlib
import os

# This notebook includes the exact Run 2 checkpoint supplied for diagnosis.
encoded = (
    '...'
)
checkpoint = base64.b64decode(encoded, validate=True)
expected_sha = "1535aee65adc1db7411ae1fc2a33622ae34ca522ce3ca6dce40b461c02ccda85"
if hashlib.sha256(checkpoint).hexdigest() != expected_sha:
    raise ValueError("Embedded checkpoint digest mismatch")
working = Path(os.environ.get("TOMAHAWK_WORK_ROOT", "/kaggle/working"))
input_folder = working / "g2_embedded_checkpoint_input"
input_folder.mkdir(parents=True, exist_ok=True)
local_checkpoint = input_folder / "Tomahawk_G2_Run_2_v2_checkpoint.zip"
local_checkpoint.write_bytes(checkpoint)
os.environ["TOMAHAWK_INPUT_ROOT"] = str(input_folder)
print("Checkpoint ready:", local_checkpoint, "SHA-256:", expected_sha, flush=True)


In [ ]:
"""Checkpoint-only geometry diagnosis for the incomplete G2 Run 2."""
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import subprocess
import sys
import traceback
import zipfile
from pathlib import Path

INPUT_NAME = "Tomahawk_G2_Run_2_v2_checkpoint.zip"
SOURCE_SHA = "86b9481410b1b0cacf00a3b20733827722742d4843b371987de92cbb0191bb8e"
ENGINE_SHA = "6bf8ea82626f6c3d485fef673ea12de6d4a5074189188bb059f8c71371fae043"
INPUT_ROOT = Path(os.environ.get("TOMAHAWK_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("TOMAHAWK_WORK_ROOT", "/kaggle/working"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPORT = WORK_ROOT / "Tomahawk_G2_Run_2_Diagnostic.json"
OUTPUT = WORK_ROOT / "Tomahawk_G2_Run_2_Diagnostic.zip"
result = {"diagnostic_version": 2, "analysis_status": "STARTED"}


def sha_bytes(data):
    return hashlib.sha256(data).hexdigest()


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(2 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


class Checkpoint:
    def __init__(self, path):
        self.path = Path(path)
        self.zip = zipfile.ZipFile(path) if self.path.is_file() else None
        if self.zip:
            names = self.zip.namelist()
            if len(names) != len(set(names)):
                raise ValueError("Checkpoint ZIP contains duplicate names")

    def read(self, member):
        if self.zip:
            return self.zip.read(member)
        path = self.path / member
        if not path.is_file():
            raise FileNotFoundError(str(path))
        return path.read_bytes()

    def close(self):
        if self.zip:
            self.zip.close()


def is_matching(checkpoint):
    try:
        status = json.loads(checkpoint.read("run_status.json"))
        return (status.get("stage") == 2
                and status.get("source_manifest_sha256") == SOURCE_SHA
                and status.get("engine_sha256") == ENGINE_SHA)
    except (KeyError, FileNotFoundError, json.JSONDecodeError, zipfile.BadZipFile):
        return False


def find_checkpoint():
    if not INPUT_ROOT.is_dir():
        raise FileNotFoundError("Kaggle Input directory is absent: " + str(INPUT_ROOT))
    exact = sorted(INPUT_ROOT.rglob(INPUT_NAME))
    renamed = sorted(
        p for p in INPUT_ROOT.rglob("Tomahawk_G2_Run_2_v2_checkpoint*.zip")
        if p not in exact
    )
    candidates = exact + renamed
    for path in sorted(INPUT_ROOT.rglob("run_status.json")):
        if path.parent not in candidates:
            candidates.append(path.parent)
    valid = []
    for path in candidates:
        try:
            check = Checkpoint(path)
            try:
                if is_matching(check):
                    valid.append(path)
            finally:
                check.close()
        except (OSError, zipfile.BadZipFile):
            pass
    if len(valid) != 1:
        visible = [str(p.relative_to(INPUT_ROOT))
                   for p in sorted(INPUT_ROOT.rglob("*"))
                   if p.is_file() and
                   (p.name.endswith(".zip") or p.name == "run_status.json")][:25]
        raise FileNotFoundError(
            "Expected exactly one Run 2 v2 checkpoint ZIP or extracted directory. "
            "Matches: " + repr([str(x) for x in valid]) +
            ". ZIP/status files found: " + repr(visible)
        )
    return valid[0]


def compare_boundary(eq, theta, zeta):
    import numpy as np
    from scipy.spatial import cKDTree
    from desc.grid import LinearGrid

    if int(eq.NFP) != 16:
        raise ValueError("Saved equilibrium has the wrong field period")
    grid = LinearGrid(rho=1.0, theta=theta, zeta=zeta, NFP=16)
    data = eq.compute(["R", "Z"], grid=grid)
    nodes = np.asarray(grid.nodes)
    model = np.column_stack(
        (np.asarray(data["R"]).reshape(-1), np.asarray(data["Z"]).reshape(-1))
    )
    if model.shape != (len(nodes), 2) or not np.isfinite(model).all():
        raise ValueError("DESC returned invalid boundary coordinates")
    t, p = nodes[:, 1], nodes[:, 2]
    reference = np.column_stack(
        (7.5 + 3.0 * np.cos(t) + 0.01 * np.cos(t - 16 * p),
         2.138969432449380 * np.sin(t) + 0.01 * np.sin(t - 16 * p))
    )
    paired = np.linalg.norm(model - reference, axis=1)
    nearest = []
    for phi in zeta:
        indices = np.flatnonzero(np.isclose(p, phi, atol=1e-9, rtol=0))
        if len(indices) != len(theta):
            raise ValueError("DESC boundary grid has unexpected angular nodes")
        a, b = model[indices], reference[indices]
        nearest.extend(cKDTree(b).query(a)[0])
        nearest.extend(cKDTree(a).query(b)[0])
    nearest = np.asarray(nearest)
    return {
        "same_theta_rms_m": float(np.sqrt(np.mean(paired**2))),
        "same_theta_max_m": float(paired.max()),
        "shape_nearest_symmetric_rms_m": float(np.sqrt(np.mean(nearest**2))),
        "shape_nearest_symmetric_max_m": float(nearest.max()),
        "R_min_m": float(model[:, 0].min()),
        "R_max_m": float(model[:, 0].max()),
        "Z_min_m": float(model[:, 1].min()),
        "Z_max_m": float(model[:, 1].max()),
    }


def diagnose():
    import numpy as np

    path = find_checkpoint()
    result["checkpoint_path"] = str(path)
    result["checkpoint_sha256"] = file_sha(path) if path.is_file() else None
    check = Checkpoint(path)
    try:
        status = json.loads(check.read("run_status.json"))
        if (status.get("stage"), status.get("complete"),
            status.get("source_manifest_sha256"),
            status.get("engine_sha256")) != (2, False, SOURCE_SHA, ENGINE_SHA):
            raise ValueError("This is not the incomplete Run 2 v2 checkpoint")
        result["checkpoint_status"] = status
        metrics = json.loads(check.read("steps/L12/metrics.json"))
        if metrics.get("resolution") != [12, 12, 3]:
            raise ValueError("Saved L12 resolution differs from expected")
        result["L12_metrics"] = metrics
        result["L16_log_tail"] = check.read("L16.log").decode(
            "utf-8", errors="replace")[-3000:]
        saved = {}
        folder = WORK_ROOT / "g2_run2_diagnostic_states"
        folder.mkdir(exist_ok=True)
        for label, member in {
            "L12_fixed": "steps/L12/fixed_equilibrium.h5",
            "L12_free": "steps/L12/free_equilibrium.h5",
            "L16_fixed": "steps/L16/fixed_equilibrium.h5",
        }.items():
            data = check.read(member)
            if not data.startswith(b"\x89HDF\r\n\x1a\n"):
                raise ValueError(member + " is not an HDF5 file")
            local = folder / (label + ".h5")
            local.write_bytes(data)
            saved[label] = local
        result["saved_state_sha256"] = {k: file_sha(v) for k, v in saved.items()}
    finally:
        check.close()

    print("Checkpoint verified. L12 optimizer success:",
          metrics.get("optimizer_success"),
          "| L16 exit:", status.get("steps", [{}])[-1].get("exit_code"),
          flush=True)
    if os.environ.get("TOMAHAWK_ARCHIVE_ONLY") == "1":
        result["analysis_status"] = "ARCHIVE_ONLY"
        return

    os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    if importlib.util.find_spec("desc") is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "desc-opt==0.17.3"]
        )
    import desc
    from desc.equilibrium import Equilibrium

    if not desc.__version__.startswith("0.17.3"):
        raise RuntimeError("DESC 0.17.3 required; found " + desc.__version__)
    result["desc_version"] = desc.__version__

    theta = np.linspace(0.0, 2 * np.pi, 128, endpoint=False)
    zeta = np.linspace(0.0, 2 * np.pi / 16, 16, endpoint=False)
    shapes, errors = {}, {}
    for label, saved_file in saved.items():
        try:
            eq = Equilibrium.load(str(saved_file))
            shapes[label] = compare_boundary(eq, theta, zeta)
            print(label, json.dumps(shapes[label], indent=2), flush=True)
            del eq
        except BaseException:
            errors[label] = traceback.format_exc()
            print(label, "could not be read; details will be in output ZIP",
                  flush=True)
    result["boundary_comparison"] = shapes
    result["state_errors"] = errors
    result["analysis_status"] = "COMPLETE" if not errors else "PARTIAL"
    result["interpretation_note"] = (
        "Compare same-theta to nearest physical R/Z distances. A small geometric "
        "distance does not establish G2 closure. Exit -9 is SIGKILL; OOM remains "
        "unconfirmed without runtime memory telemetry."
    )


try:
    diagnose()
except BaseException as error:
    result["analysis_status"] = "FAILED"
    result["error"] = str(error)
    result["traceback"] = traceback.format_exc()
finally:
    REPORT.write_text(json.dumps(result, indent=2, allow_nan=False))
    with zipfile.ZipFile(OUTPUT, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(REPORT, REPORT.name)
        if "L16_log_tail" in result:
            archive.writestr("L16_log_tail.txt", result["L16_log_tail"])
    print("DOWNLOAD THIS FILE EVEN IF A STEP FAILED:", OUTPUT, flush=True)
    print("STATUS:", result["analysis_status"], flush=True)
    if "error" in result:
        print(result["traceback"][-4500:], flush=True)
